# Qwen Coder worker on Google Colab

This notebook starts a small Qwen Coder model behind an OpenAI-compatible FastAPI server. It intentionally avoids vLLM for the verification path: the HTTP port opens immediately, `/health` reports `loading` or `ready`, and the model loads in the background.

In [ ]:
import os, subprocess, json, psutil, getpass, pathlib, time, socket, requests

def sh(cmd, check=True):
    return subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=check).stdout

gpu = sh('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || true', check=False).strip()
ram_gb = round(psutil.virtual_memory().total / (1024**3), 1)
print('GPU:', gpu or 'NO GPU')
print('RAM_GB:', ram_gb)
if not gpu:
    raise RuntimeError('No GPU detected. Enable GPU runtime before continuing.')


In [ ]:
# Install the lightweight serving stack. Restart runtime if Colab asks for it, then rerun from the top.
!pip install -U "transformers>=4.46.0" "accelerate>=0.34.0" fastapi uvicorn requests pyngrok


In [ ]:
# Stop any stale server process and clear the API port.
SERVER_PORT = int(os.environ.get('SERVER_PORT', '8000'))
if 'proc' in globals() and proc.poll() is None:
    print('Terminating previous proc:', proc.pid)
    proc.terminate()
    try:
        proc.wait(timeout=20)
    except subprocess.TimeoutExpired:
        print('Killing previous proc:', proc.pid)
        proc.kill()
        proc.wait(timeout=10)

print(sh("pkill -f 'qwen_openai_server.py' || true", check=False))
print(sh("pkill -f 'vllm.entrypoints.openai.api_server' || true", check=False))
print(sh(f"fuser -k {SERVER_PORT}/tcp 2>/dev/null || true", check=False))
time.sleep(3)
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.settimeout(2)
    port_open_before_start = s.connect_ex(('127.0.0.1', SERVER_PORT)) == 0
print('port_open_before_start:', port_open_before_start)
if port_open_before_start:
    raise RuntimeError(f'Port {SERVER_PORT} is still occupied after cleanup.')


In [ ]:
MODEL_ID = os.environ.get('QWEN_MODEL_ID', 'Qwen/Qwen2.5-Coder-1.5B-Instruct')
MAX_NEW_TOKENS = int(os.environ.get('MAX_NEW_TOKENS', '512'))
API_KEY = os.environ.get('QWEN_COLAB_API_KEY') or getpass.getpass('Set QWEN_COLAB_API_KEY for this Colab session: ')
os.environ['QWEN_MODEL_ID'] = MODEL_ID
os.environ['QWEN_COLAB_API_KEY'] = API_KEY
print(json.dumps({'selected_model': MODEL_ID, 'max_new_tokens': MAX_NEW_TOKENS, 'port': SERVER_PORT}, indent=2))


In [ ]:
server_py = pathlib.Path('/content/qwen_openai_server.py')
server_py.write_text(r'''
import os, time, uuid, threading, traceback, json
from typing import Any

import torch
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = os.environ.get('QWEN_MODEL_ID', 'Qwen/Qwen2.5-Coder-1.5B-Instruct')
API_KEY = os.environ.get('QWEN_COLAB_API_KEY', '')
MAX_NEW_TOKENS_DEFAULT = int(os.environ.get('MAX_NEW_TOKENS', '512'))

app = FastAPI(title='qwen-colab-openai-compatible-server')
state: dict[str, Any] = {'status': 'loading', 'error': None, 'started_at': time.time()}
tokenizer = None
model = None

class AnyPayload(BaseModel):
    model: str | None = None
    messages: list[dict[str, Any]] | None = None
    input: Any | None = None
    instructions: str | None = None
    max_tokens: int | None = None
    max_output_tokens: int | None = None
    temperature: float | None = None
    stream: bool | None = None

def require_auth(authorization: str | None):
    if API_KEY and authorization != f'Bearer {API_KEY}':
        raise HTTPException(status_code=401, detail='Unauthorized')

def load_model():
    global tokenizer, model
    try:
        print(f'Loading model: {MODEL_ID}', flush=True)
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            torch_dtype=dtype,
            device_map='auto',
            trust_remote_code=True,
            low_cpu_mem_usage=True,
        )
        model.eval()
        state['status'] = 'ready'
        print('Model ready', flush=True)
    except Exception:
        state['status'] = 'error'
        state['error'] = traceback.format_exc()
        print(state['error'], flush=True)

threading.Thread(target=load_model, daemon=True).start()

def gpu_state():
    if not torch.cuda.is_available():
        return {'cuda': False}
    return {
        'cuda': True,
        'name': torch.cuda.get_device_name(0),
        'memory_allocated_mb': round(torch.cuda.memory_allocated(0) / 1024 / 1024, 1),
        'memory_reserved_mb': round(torch.cuda.memory_reserved(0) / 1024 / 1024, 1),
    }

@app.get('/health')
def health():
    return {'status': state['status'], 'error': state['error'], 'model': MODEL_ID, 'gpu': gpu_state()}

@app.get('/v1/models')
def models(authorization: str | None = Header(default=None)):
    require_auth(authorization)
    return {'object': 'list', 'data': [{'id': MODEL_ID, 'object': 'model', 'owned_by': 'qwen-colab'}]}

def ensure_ready():
    if state['status'] == 'error':
        raise HTTPException(status_code=500, detail=state['error'])
    if state['status'] != 'ready':
        raise HTTPException(status_code=503, detail='Model is still loading')

def messages_from_input(payload: AnyPayload):
    if payload.messages:
        return payload.messages
    messages = []
    if payload.instructions:
        messages.append({'role': 'system', 'content': payload.instructions})
    if isinstance(payload.input, str):
        messages.append({'role': 'user', 'content': payload.input})
    elif isinstance(payload.input, list):
        for item in payload.input:
            if isinstance(item, dict):
                messages.append({'role': item.get('role', 'user'), 'content': item.get('content') or item.get('text') or ''})
    if not messages:
        messages.append({'role': 'user', 'content': ''})
    return messages

def generate_text(messages, max_new_tokens, temperature):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    do_sample = temperature is not None and temperature > 0
    generation_kwargs = {
        'max_new_tokens': max_new_tokens,
        'do_sample': do_sample,
        'pad_token_id': tokenizer.eos_token_id,
    }
    if do_sample:
        generation_kwargs['temperature'] = temperature
    with torch.inference_mode():
        output = model.generate(**inputs, **generation_kwargs)
    generated = output[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def sse(event: str, data: dict[str, Any]):
    return f'event: {event}\ndata: {json.dumps(data, ensure_ascii=False)}\n\n'

def stream_chat_response(response: dict[str, Any], text: str):
    chunk_id = response['id']
    model_name = response['model']
    created = response['created']
    yield 'data: ' + json.dumps({'id': chunk_id, 'object': 'chat.completion.chunk', 'created': created, 'model': model_name, 'choices': [{'index': 0, 'delta': {'role': 'assistant'}, 'finish_reason': None}]}, ensure_ascii=False) + '\n\n'
    yield 'data: ' + json.dumps({'id': chunk_id, 'object': 'chat.completion.chunk', 'created': created, 'model': model_name, 'choices': [{'index': 0, 'delta': {'content': text}, 'finish_reason': None}]}, ensure_ascii=False) + '\n\n'
    yield 'data: ' + json.dumps({'id': chunk_id, 'object': 'chat.completion.chunk', 'created': created, 'model': model_name, 'choices': [{'index': 0, 'delta': {}, 'finish_reason': 'stop'}]}, ensure_ascii=False) + '\n\n'
    yield 'data: [DONE]\n\n'

def stream_response_response(response: dict[str, Any], text: str):
    response_id = response['id']
    model_name = response['model']
    created_at = response['created_at']
    item_id = 'msg_' + uuid.uuid4().hex
    output_index = 0
    content_index = 0
    yield sse('response.created', {'type': 'response.created', 'response': {'id': response_id, 'object': 'response', 'created_at': created_at, 'model': model_name, 'status': 'in_progress', 'output': []}})
    yield sse('response.output_item.added', {'type': 'response.output_item.added', 'output_index': output_index, 'item': {'id': item_id, 'type': 'message', 'status': 'in_progress', 'role': 'assistant', 'content': []}})
    yield sse('response.content_part.added', {'type': 'response.content_part.added', 'item_id': item_id, 'output_index': output_index, 'content_index': content_index, 'part': {'type': 'output_text', 'text': ''}})
    yield sse('response.output_text.delta', {'type': 'response.output_text.delta', 'item_id': item_id, 'output_index': output_index, 'content_index': content_index, 'delta': text})
    yield sse('response.output_text.done', {'type': 'response.output_text.done', 'item_id': item_id, 'output_index': output_index, 'content_index': content_index, 'text': text})
    yield sse('response.content_part.done', {'type': 'response.content_part.done', 'item_id': item_id, 'output_index': output_index, 'content_index': content_index, 'part': {'type': 'output_text', 'text': text}})
    yield sse('response.output_item.done', {'type': 'response.output_item.done', 'output_index': output_index, 'item': {'id': item_id, 'type': 'message', 'status': 'completed', 'role': 'assistant', 'content': [{'type': 'output_text', 'text': text}]}})
    yield sse('response.completed', {'type': 'response.completed', 'response': response})
    yield 'data: [DONE]\n\n'

@app.post('/v1/chat/completions')
def chat(payload: AnyPayload, authorization: str | None = Header(default=None)):
    require_auth(authorization)
    ensure_ready()
    messages = messages_from_input(payload)
    text = generate_text(messages, payload.max_tokens or MAX_NEW_TOKENS_DEFAULT, payload.temperature)
    response = {
        'id': 'chatcmpl_' + uuid.uuid4().hex,
        'object': 'chat.completion',
        'created': int(time.time()),
        'model': payload.model or MODEL_ID,
        'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': text}, 'finish_reason': 'stop'}],
    }
    if payload.stream:
        return StreamingResponse(stream_chat_response(response, text), media_type='text/event-stream')
    return response

@app.post('/v1/responses')
def responses(payload: AnyPayload, authorization: str | None = Header(default=None)):
    require_auth(authorization)
    ensure_ready()
    messages = messages_from_input(payload)
    text = generate_text(messages, payload.max_output_tokens or payload.max_tokens or MAX_NEW_TOKENS_DEFAULT, payload.temperature)
    response = {
        'id': 'resp_' + uuid.uuid4().hex,
        'object': 'response',
        'created_at': int(time.time()),
        'model': payload.model or MODEL_ID,
        'output': [{'type': 'message', 'role': 'assistant', 'content': [{'type': 'output_text', 'text': text}]}],
        'output_text': text,
    }
    if payload.stream:
        return StreamingResponse(stream_response_response(response, text), media_type='text/event-stream')
    return response
''', encoding='utf-8')
print('Wrote', server_py)


In [ ]:
# Start the server. The port should open quickly; /health will show loading until the model is ready.
log_path = pathlib.Path('/content/qwen_server.log')
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
cmd = ['python', '-u', str(server_py)]
uvicorn_cmd = ['python', '-u', '-m', 'uvicorn', 'qwen_openai_server:app', '--host', '0.0.0.0', '--port', str(SERVER_PORT), '--app-dir', '/content']
print('Starting:', ' '.join(uvicorn_cmd))
log_file = open(log_path, 'w', buffering=1)
proc = subprocess.Popen(uvicorn_cmd, stdout=log_file, stderr=subprocess.STDOUT, env=env, text=True)
print('PID:', proc.pid, 'log:', log_path)
time.sleep(3)


In [ ]:
# Diagnostics. If port_open is False here, the server process failed before binding the port.
print('process_poll:', proc.poll())
print('gpu_state:')
print(sh('nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader', check=False))
print('log_tail:')
if log_path.exists():
    print('\n'.join(log_path.read_text(errors='ignore').splitlines()[-120:]))
else:
    print('No log file yet:', log_path)

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.settimeout(2)
    port_open = s.connect_ex(('127.0.0.1', SERVER_PORT)) == 0
print('port_open:', port_open)


In [ ]:
# Wait for /health status=ready, then run a minimal chat request.
base = f'http://127.0.0.1:{SERVER_PORT}'
ready = False
for i in range(120):
    if proc.poll() is not None:
        tail = '\n'.join(log_path.read_text(errors='ignore').splitlines()[-160:]) if log_path.exists() else ''
        raise RuntimeError(f'Server exited before readiness. Exit code={proc.returncode}. Log tail:\n{tail}')
    try:
        r = requests.get(base + '/health', timeout=5)
        data = r.json()
        print('health', data.get('status'), data.get('gpu'))
        if data.get('status') == 'ready':
            ready = True
            break
        if data.get('status') == 'error':
            raise RuntimeError(data.get('error'))
    except Exception as e:
        if i % 6 == 0:
            print('waiting', i, type(e).__name__, str(e)[:200])
    time.sleep(10)

if not ready:
    tail = '\n'.join(log_path.read_text(errors='ignore').splitlines()[-160:]) if log_path.exists() else ''
    raise TimeoutError(f'Model did not become ready. Log tail:\n{tail}')

payload = {'model': MODEL_ID, 'messages': [{'role': 'user', 'content': 'Return exactly: ok'}], 'max_tokens': 16}
r = requests.post(base + '/v1/chat/completions', headers={'Authorization': f'Bearer {API_KEY}'}, json=payload, timeout=120)
print(r.status_code)
print(r.text[:1000])


In [ ]:
# Preferred tunnel: Cloudflare Tunnel without token.
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
import re
cf_log = pathlib.Path('/content/cloudflared.log')
cf = subprocess.Popen(['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{SERVER_PORT}', '--no-autoupdate'], stdout=open(cf_log, 'w'), stderr=subprocess.STDOUT, text=True)
print('cloudflared PID:', cf.pid, 'log:', cf_log)
time.sleep(8)
text = cf_log.read_text(errors='ignore')
print(text[-2000:])
match = re.search(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com', text)
if match:
    public_url = match.group(0)
    print('\nSet locally:')
    print('QWEN_COLAB_BASE_URL=' + public_url)
    print('QWEN_COLAB_API_KEY=<the key you entered>')
    print('QWEN_MODEL_ID=' + MODEL_ID)
else:
    print('Tunnel URL not found yet. Rerun this cell or inspect /content/cloudflared.log')


In [ ]:
# Optional ngrok tunnel. Requires NGROK_AUTHTOKEN in the Colab environment.
if os.environ.get('NGROK_AUTHTOKEN'):
    from pyngrok import ngrok
    ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])
    tunnel = ngrok.connect(SERVER_PORT, 'http')
    print('NGROK URL:', tunnel.public_url)
else:
    print('Skipping ngrok: set NGROK_AUTHTOKEN to use this option.')


In [ ]:
# Probe /v1/responses. If this works, Codex can use the Colab URL directly.
payload = {'model': MODEL_ID, 'input': 'Return exactly: ok', 'max_output_tokens': 16}
try:
    r = requests.post(base + '/v1/responses', headers={'Authorization': f'Bearer {API_KEY}'}, json=payload, timeout=120)
    print(r.status_code, r.text[:1000])
except Exception as e:
    print('Responses endpoint unavailable.', repr(e))
